In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../backend")

In [3]:
import torch as t
import torch.nn as nn
device = "cuda" if t.cuda.is_available() else "mps" if t.backends.mps.is_available() else "cpu"
device

'cuda'

In [4]:
import pandas as pd
import torch as t
import torch.nn as nn
import torch.optim as optim
import numpy as np
from cuml.manifold.umap import UMAP
from cuml import TSNE

In [5]:
data=pd.read_table("../data/datasets/scivis/alloy_data.txt")
data.describe()

,KS1295[%],6082[%],2024[%],bat-box[%],3003[%],4032[%],Al,Si,Cu,Ni,...,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136
count,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,16.500000,16.500000,16.500000,16.500000,16.500000,17.500000,91.024760,4.710985,1.633425,0.569050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,15.276055,15.276055,15.276055,15.276055,15.276055,15.276055,2.835679,2.273711,0.730324,0.329574,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,79.966460,0.716500,0.058500,0.013000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,3.300000,3.300000,3.300000,3.300000,3.300000,4.300000,89.113631,2.873941,1.072590,0.311980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,13.200000,13.200000,13.200000,13.200000,13.200000,14.200000,91.267046,4.399894,1.554225,0.527800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,23.100000,23.100000,23.100000,23.100000,23.100000,24.100000,93.164818,6.259213,2.118360,0.782725,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,99.000000,99.000000,99.000000,99.000000,99.000000,100.000000,97.168700,12.695500,4.612500,2.012800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
data

,KS1295[%],6082[%],2024[%],bat-box[%],3003[%],4032[%],Al,Si,Cu,Ni,...,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136
0,0.0,0.0,0.0,0.0,0.0,100.0,83.675000,12.250000,0.900000,1.300000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,0.0,0.0,0.0,3.3,96.7,84.118850,11.865550,0.874425,1.257100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,0.0,0.0,6.6,93.4,84.562700,11.481100,0.848850,1.214200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,0.0,0.0,0.0,9.9,90.1,85.006550,11.096650,0.823275,1.171300,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,0.0,0.0,0.0,13.2,86.8,85.450400,10.712200,0.797700,1.128400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324627,95.7,0.0,0.0,0.0,3.3,1.0,80.533928,12.296200,3.381765,1.946140,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
324628,95.7,0.0,0.0,3.3,0.0,1.0,80.539868,12.297322,3.397440,1.946140,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
324629,95.7,0.0,3.3,0.0,0.0,1.0,80.521553,12.309400,3.379290,1.946965,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
324630,95.7,3.3,0.0,0.0,0.0,1.0,80.358533,12.297025,3.531090,1.946965,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print(data.columns.to_list())

['KS1295[%]', '6082[%]', '2024[%]', 'bat-box[%]', '3003[%]', '4032[%]', 'Al', 'Si', 'Cu', 'Ni', 'Mg', 'Mn', 'Fe', 'Cr', 'Ti', 'Zr', 'V', 'Zn', 'Vf_FCC_A1', 'Vf_DIAMOND_A4', 'Vf_AL15SI2M4', 'Vf_AL3X', 'Vf_AL6MN', 'Vf_MG2ZN3', 'Vf_AL3NI2', 'Vf_AL3NI_D011', 'Vf_AL7CU4NI', 'Vf_AL2CU_C16', 'Vf_Q_ALCUMGSI', 'Vf_AL7CU2FE', 'Vf_MG2SI_C1', 'Vf_AL9FE2SI2', 'Vf_AL18FE2MG7SI10', 'eut. frac.[%]', 'eut. T (�C)', 'T_FCC_A1', 'T_DIAMOND_A4', 'T_AL15SI2M4', 'T_AL3X', 'T_AL6MN', 'T_MG2ZN3', 'T_AL3NI2', 'T_AL3NI_D011', 'T_AL7CU4NI', 'T_AL2CU_C16', 'T_Q_ALCUMGSI', 'T_AL7CU2FE', 'T_MG2SI_C1', 'T_AL9FE2SI2', 'T_AL18FE2MG7SI10', 'T(liqu)', 'T(sol)', 'delta_T', 'delta_T_FCC', 'delta_T_Al15Si2M4', 'delta_T_Si', 'CSC', 'YS(MPa)', 'hardness(Vickers)', 'CTEvol(1/K)(20.0-300.0�C)', 'Density(g/cm3)', 'Volume(m3/mol)', 'El.conductivity(S/m)', 'El. resistivity(ohm m)', 'heat capacity(J/(mol K))', 'Therm.conductivity(W/(mK))', 'Therm. diffusivity(m2/s)', 'Therm.resistivity(mK/W)', 'Linear thermal expansion (1/K)(20.0-

In [8]:
data.describe()

,KS1295[%],6082[%],2024[%],bat-box[%],3003[%],4032[%],Al,Si,Cu,Ni,...,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136
count,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,16.500000,16.500000,16.500000,16.500000,16.500000,17.500000,91.024760,4.710985,1.633425,0.569050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,15.276055,15.276055,15.276055,15.276055,15.276055,15.276055,2.835679,2.273711,0.730324,0.329574,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,79.966460,0.716500,0.058500,0.013000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,3.300000,3.300000,3.300000,3.300000,3.300000,4.300000,89.113631,2.873941,1.072590,0.311980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,13.200000,13.200000,13.200000,13.200000,13.200000,14.200000,91.267046,4.399894,1.554225,0.527800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,23.100000,23.100000,23.100000,23.100000,23.100000,24.100000,93.164818,6.259213,2.118360,0.782725,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,99.000000,99.000000,99.000000,99.000000,99.000000,100.000000,97.168700,12.695500,4.612500,2.012800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
input_cols= data.columns.to_list()[:6]
output_cols= data.columns.to_list()[6:70]
input_cols, output_cols

(['KS1295[%]', '6082[%]', '2024[%]', 'bat-box[%]', '3003[%]', '4032[%]'],
 ['Al',
  'Si',
  'Cu',
  'Ni',
  'Mg',
  'Mn',
  'Fe',
  'Cr',
  'Ti',
  'Zr',
  'V',
  'Zn',
  'Vf_FCC_A1',
  'Vf_DIAMOND_A4',
  'Vf_AL15SI2M4',
  'Vf_AL3X',
  'Vf_AL6MN',
  'Vf_MG2ZN3',
  'Vf_AL3NI2',
  'Vf_AL3NI_D011',
  'Vf_AL7CU4NI',
  'Vf_AL2CU_C16',
  'Vf_Q_ALCUMGSI',
  'Vf_AL7CU2FE',
  'Vf_MG2SI_C1',
  'Vf_AL9FE2SI2',
  'Vf_AL18FE2MG7SI10',
  'eut. frac.[%]',
  'eut. T (�C)',
  'T_FCC_A1',
  'T_DIAMOND_A4',
  'T_AL15SI2M4',
  'T_AL3X',
  'T_AL6MN',
  'T_MG2ZN3',
  'T_AL3NI2',
  'T_AL3NI_D011',
  'T_AL7CU4NI',
  'T_AL2CU_C16',
  'T_Q_ALCUMGSI',
  'T_AL7CU2FE',
  'T_MG2SI_C1',
  'T_AL9FE2SI2',
  'T_AL18FE2MG7SI10',
  'T(liqu)',
  'T(sol)',
  'delta_T',
  'delta_T_FCC',
  'delta_T_Al15Si2M4',
  'delta_T_Si',
  'CSC',
  'YS(MPa)',
  'hardness(Vickers)',
  'CTEvol(1/K)(20.0-300.0�C)',
  'Density(g/cm3)',
  'Volume(m3/mol)',
  'El.conductivity(S/m)',
  'El. resistivity(ohm m)',
  'heat capacity(J/(mol K))',
  

In [10]:
from col_defs import *

In [11]:
cleaned = data[input_cols + output_cols].fillna(0)
cleaned

,KS1295[%],6082[%],2024[%],bat-box[%],3003[%],4032[%],Al,Si,Cu,Ni,...,Density(g/cm3),Volume(m3/mol),El.conductivity(S/m),El. resistivity(ohm m),heat capacity(J/(mol K)),Therm.conductivity(W/(mK)),Therm. diffusivity(m2/s),Therm.resistivity(mK/W),Linear thermal expansion (1/K)(20.0-300.0�C),Technical thermal expansion (1/K)(20.0-300.0�C)
0,0.0,0.0,0.0,0.0,0.0,100.0,83.675000,12.250000,0.900000,1.300000,...,2.65803,0.00001,11302200,8.851450e-08,27.3373,159.046,0.000060,0.006288,0.000024,0.000022
1,0.0,0.0,0.0,0.0,3.3,96.7,84.118850,11.865550,0.874425,1.257100,...,2.65879,0.00001,11414800,8.767350e-08,27.3430,160.429,0.000061,0.006232,0.000024,0.000022
2,0.0,0.0,0.0,0.0,6.6,93.4,84.562700,11.481100,0.848850,1.214200,...,2.65935,0.00001,11489900,8.708070e-08,27.3633,161.346,0.000061,0.006203,0.000024,0.000022
3,0.0,0.0,0.0,0.0,9.9,90.1,85.006550,11.096650,0.823275,1.171300,...,2.66111,0.00001,11566400,8.646380e-08,27.3806,162.105,0.000061,0.006168,0.000024,0.000022
4,0.0,0.0,0.0,0.0,13.2,86.8,85.450400,10.712200,0.797700,1.128400,...,2.66318,0.00001,11650800,8.581640e-08,27.3873,163.127,0.000061,0.006127,0.000024,0.000022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324627,95.7,0.0,0.0,0.0,3.3,1.0,80.533928,12.296200,3.381765,1.946140,...,2.72509,0.00001,10539500,9.491770e-08,27.3289,149.812,0.000056,0.006691,0.000023,0.000021
324628,95.7,0.0,0.0,3.3,0.0,1.0,80.539868,12.297322,3.397440,1.946140,...,2.72496,0.00001,10545800,9.494870e-08,27.3291,149.752,0.000056,0.006688,0.000023,0.000021
324629,95.7,0.0,3.3,0.0,0.0,1.0,80.521553,12.309400,3.379290,1.946965,...,2.72497,0.00001,10558200,9.485710e-08,27.3189,149.799,0.000056,0.006686,0.000023,0.000021
324630,95.7,3.3,0.0,0.0,0.0,1.0,80.358533,12.297025,3.531090,1.946965,...,2.72756,0.00001,10552100,9.486620e-08,27.3293,149.771,0.000056,0.006683,0.000023,0.000021


In [12]:
import noise
import models

In [13]:
import torch.utils.data as data_utils
from tqdm import tqdm


In [14]:
def train_vae(
    cleaned: pd.DataFrame,
    vae_model=models.VAE,
    noise_model: noise.Noiser = None,
    val_percent: float = 0.2,
    test_percent: float = 0.1,
    epochs: int = 3,
):
    diff = cleaned.max() - cleaned.min()
    diff[diff == 0] = cleaned.max()[diff == 0]
    diff[diff == 0] = 1
    data_normed = (cleaned - cleaned.min()) / diff
    data_t = t.tensor(data_normed[input_cols + output_cols].values, dtype=t.float32)
    noise_t = t.zeros_like(data_t)
    if noise_model is not None:
        data_t, noise_t = noise_model.add_noise(data_t)
        print("Added noise with model:", noise_model.__class__.__name__)

    dataset = data_utils.TensorDataset(data_t, noise_t)

    val_size = int(len(dataset) * val_percent)
    test_size = int(len(dataset) * test_percent)
    train_size = len(dataset) - val_size - test_size
    train_dataset, val_dataset, test_dataset = data_utils.random_split(
        dataset, [train_size, val_size, test_size]
    )

    train_dataloader = data_utils.DataLoader(
        train_dataset, batch_size=128, shuffle=True
    )
    val_dataloader = data_utils.DataLoader(val_dataset, batch_size=128, shuffle=False)
    model: nn.Module = vae_model(
        input_size=len(input_cols) + len(output_cols),
        layers=[256, 64],
        latent_size=16,
        norm=nn.BatchNorm1d,
        out_norm=nn.BatchNorm1d,
    ).to(device)
    optimizer = t.optim.Adam(model.parameters(), lr=0.001)

    # log mse loss
    def log_mse_loss(recon_x, x):
        mse_loss = nn.MSELoss()(recon_x, x)
        return t.log(mse_loss + 1e-8)

    def vae_loss(recon_x, x, mu, logvar) -> t.Tensor:
        mse_loss = nn.MSELoss()(recon_x, x)
        KLD = -0.5 * t.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return mse_loss + KLD / x.size(0)

    model.train()
    loss_train = []
    loss_val = []
    for epoch in range(epochs):
        gen = tqdm(train_dataloader)
        for i, batch_inputs in enumerate(gen):
            batch_inputs = batch_inputs[0].to(device)
            optimizer.zero_grad()
            outputs, mu, logvar = model(batch_inputs)
            loss = vae_loss(outputs, batch_inputs, mu, logvar)
            loss.backward()
            optimizer.step()
            loss_train.append(loss.item())
            gen.set_description(
                f"Epoch {epoch + 1}/{epochs} | Batch {i + 1}/{len(train_dataloader)} | Train Loss: {loss.item():.4f} "
            )
        loss_val = []
        for val_batch in val_dataloader:
            val_inputs = val_batch[0].to(device)
            with t.no_grad():
                val_outputs, val_mu, val_logvar = model(val_inputs)
                val_loss = vae_loss(val_outputs, val_inputs, val_mu, val_logvar)
                loss_val.append(val_loss.item())
        avg_val_loss = np.mean(loss_val)
        print(
            f"Epoch {epoch + 1}/{epochs}, Train Loss: {loss.item():.4f}, Val Loss: {avg_val_loss:.4f}"
        )
    model.eval()
    return {
        "base_data": data_t,
        "noise_data": noise_t,
        "vae_model": model,
        "training": {
            "data": train_dataset[:][0],
            "noise": train_dataset[:][1],
        },
        "validation": {
            "data": val_dataset[:][0],
            "noise": val_dataset[:][1],
        },
        "test": {
            "data": test_dataset[:][0],
            "noise": test_dataset[:][1],
        },
    }


# train_vae(cleaned)

In [15]:
results = train_vae(
    cleaned,
    vae_model=models.VAE,
    noise_model=noise.ColumnBasedNoiser(noise_level=0.1),
    epochs=5,
)

Added noise with model: ColumnBasedNoiser


Epoch 1/5 | Batch 1776/1776 | Train Loss: 0.4966 : 100%|██████████| 1776/1776 [00:03<00:00, 515.92it/s]


Epoch 1/5, Train Loss: 0.4966, Val Loss: 0.3249


Epoch 2/5 | Batch 1776/1776 | Train Loss: 0.0525 : 100%|██████████| 1776/1776 [00:03<00:00, 526.95it/s]


Epoch 2/5, Train Loss: 0.0525, Val Loss: 0.0480


...

Epoch 3/5, Train Loss: 0.0287, Val Loss: 0.0302


Epoch 4/5 | Batch 1776/1776 | Train Loss: 0.0276 : 100%|██████████| 1776/1776 [00:03<00:00, 535.87it/s]


Epoch 4/5, Train Loss: 0.0276, Val Loss: 0.0295


Epoch 5/5 | Batch 1776/1776 | Train Loss: 0.0280 : 100%|██████████| 1776/1776 [00:03<00:00, 576.32it/s]


Epoch 5/5, Train Loss: 0.0280, Val Loss: 0.0293


In [19]:
from torchmetrics.functional import f1_score, r2_score
from torchmetrics.functional.retrieval import retrieval_normalized_dcg, retrieval_precision, retrieval_recall
from torchmetrics.retrieval import recall

import models.base


def noise_metric_r2(noise: t.Tensor, predicted_noise: t.Tensor) -> float:
    return r2_score(noise.flatten(), predicted_noise.flatten()).item()


def noise_metric_ndcg_k(
    noise: t.Tensor, predicted_noise: t.Tensor, k: int = 10
) -> float:
    from torchmetrics.retrieval import RetrievalNormalizedDCG as NDCG
    max_noises_target = t.argsort(noise, descending=True, dim=1)
    probs_noises_target = max_noises_target.float() / float(noise.shape[-1])
    max_noises_predicted = t.argsort(predicted_noise, descending=True, dim=1)
    probs_noises_predicted = max_noises_predicted.float() / float(noise.shape[-1])
    
    return retrieval_normalized_dcg(probs_noises_target, probs_noises_predicted, top_k=k).mean().item()


def noise_metric_f1_k(noise: t.Tensor, predicted_noise: t.Tensor, k: int = 10) -> float:
    max_noises_target = t.argsort(noise, descending=True, dim=1)
    probs_noises_target = max_noises_target.float() / float(noise.shape[-1])
    max_noises_predicted = t.argsort(predicted_noise, descending=True, dim=1)
    probs_noises_predicted = max_noises_predicted.float() / float(noise.shape[-1])
    precision = retrieval_precision(probs_noises_target, probs_noises_predicted, top_k=k)
    recall_score = retrieval_recall(probs_noises_target, probs_noises_predicted, top_k=k)
    
    return (2 * precision * recall_score / (precision + recall_score + 1e-8)).mean().item()
def noise_metric_precision_k(noise: t.Tensor, predicted_noise: t.Tensor, k: int = 10) -> float:
    max_noises_target = t.argsort(noise, descending=True, dim=1)
    probs_noises_target = max_noises_target.float() / float(noise.shape[-1])
    max_noises_predicted = t.argsort(predicted_noise, descending=True, dim=1)
    probs_noises_predicted = max_noises_predicted.float() / float(noise.shape[-1])
    
    return retrieval_precision(probs_noises_target, probs_noises_predicted, top_k=k).mean().item()

def noise_metric_recall_k(noise: t.Tensor, predicted_noise: t.Tensor, k: int = 10) -> float:
    max_noises_target = t.argsort(noise, descending=True, dim=1)
    probs_noises_target = max_noises_target.float() / float(noise.shape[-1])
    max_noises_predicted = t.argsort(predicted_noise, descending=True, dim=1)
    probs_noises_predicted = max_noises_predicted.float() / float(noise.shape[-1])
    
    return retrieval_recall(probs_noises_target, probs_noises_predicted, top_k=k).mean().item()

metrics = {
    "R2": noise_metric_r2,
    "NDCG@10": noise_metric_ndcg_k,
    "F1@10": noise_metric_f1_k,
    "Precision@10": noise_metric_precision_k,
    "Recall@10": noise_metric_recall_k,
}
model: models.base.BaseUCQModel = results["vae_model"]
for split in ["training", "validation", "test"]:
    data = results[split]["data"].to(device)
    noise_data = results[split]["noise"].to(device)
    with t.no_grad():
        reconstructed, _, _ = model(data)
        uq = model.uncertainty(data)
        uq_reconstructed = model.uncertainty(reconstructed)
    print(
        "R2 score for reconstruction:",
        r2_score(data.flatten(), reconstructed.flatten()).item(),
    )
    print(f"UQ Metrics for {split} set:")
    for metric_name, metric_fn in metrics.items():
        metric_value = metric_fn(noise_data, uq_reconstructed)
        print(f"  {metric_name}: {metric_value:.4f}")

R2 score for reconstruction: 0.4987565875053406
UQ Metrics for training set:
  R2: -73.3700
  NDCG@10: 0.5830
  F1@10: 0.0000
  Precision@10: 0.5414
  Recall@10: 0.0000
R2 score for reconstruction: 0.49387240409851074
UQ Metrics for validation set:
  R2: -73.3843
  NDCG@10: 0.5834
  F1@10: 0.0000
  Precision@10: 0.6400
  Recall@10: 0.0000
R2 score for reconstruction: 0.5024539232254028
UQ Metrics for test set:
  R2: -73.4687
  NDCG@10: 0.5821
  F1@10: 0.0000
  Precision@10: 0.5814
  Recall@10: 0.0000


In [17]:
results["training"]["noise"]

tensor([[0.0920, 0.0954, 0.0399,  ..., 0.0212, 0.0210, 0.0210],
        [0.0920, 0.0954, 0.0399,  ..., 0.0212, 0.0210, 0.0210],
        [0.0920, 0.0954, 0.0399,  ..., 0.0212, 0.0210, 0.0210],
        ...,
        [0.0920, 0.0954, 0.0399,  ..., 0.0212, 0.0210, 0.0210],
        [0.0920, 0.0954, 0.0399,  ..., 0.0212, 0.0210, 0.0210],
        [0.0920, 0.0954, 0.0399,  ..., 0.0212, 0.0210, 0.0210]])